# Series by computer: the crystal, a series inside a series, Taylor about a point

Today the computer does the bookkeeping. Three sections, in the order we use them in class:

1. **The crystal.** Friday we claimed the bracket $\dfrac{1}{(d+x)^2} - \dfrac{1}{(d-x)^2} \approx -\dfrac{4x}{d^3}$. SymPy writes out more terms than we need; we cut at the linear term and plot.
2. **A series inside a series.** $\dfrac{1}{1+\sin x}$ through $x^3$, by putting the sine series into $\dfrac{1}{1+a}$.
3. **Taylor about a point.** Approximating $\cos x$ near $x = 3.5$: centered at 0, or centered at 3?

Run each cell with Shift-Enter.

## Predict first

Double-click the cell below, type your guesses, then Shift-Enter.

1. The crystal bracket: after the $-4x/d^3$ term, is the next term $x^2$ or $x^3$? Why?
2. $\dfrac{1}{1+\sin x} = 1 - x + c_2 x^2 + c_3 x^3 + \dots$: what is $c_2$?
3. To get $\cos(3.5)$ within 1%: how many non-zero terms of the series about $x = 0$? About $x = 3$?

*(your predictions here)*

In [ ]:
%matplotlib inline
import numpy as np
from matplotlib import pyplot as plt
import sympy as sp
from ipywidgets import interact

x, a, u = sp.symbols('x a u')

## 1. The crystal

Write the bracket in terms of $u = x/d$, which is small when $x$ is small compared with $d$. Then the bracket is
$\dfrac{1}{d^2}\left[\dfrac{1}{(1+u)^2} - \dfrac{1}{(1-u)^2}\right]$, and the part in square brackets has no $d$ in it.

In [ ]:
bracket = 1/(1 + u)**2 - 1/(1 - u)**2
bracket.series(u, 0, 8)        # every term through u^7; O(u^8) is what is left out

Cut it off at the linear term (and, for comparison, at the cubic term):

In [ ]:
linear = bracket.series(u, 0, 2).removeO()
cubic = bracket.series(u, 0, 4).removeO()
linear, cubic

Plot the bracket and both approximations. Where does the linear one stop being good?

In [ ]:
us = np.linspace(-0.5, 0.5, 400)
plt.figure()
plt.plot(us, sp.lambdify(u, bracket)(us), 'k--', label='the bracket')
plt.plot(us, sp.lambdify(u, linear)(us), 'red', label=f'linear: ${sp.latex(linear)}$')
plt.plot(us, sp.lambdify(u, cubic)(us), 'purple', label=f'through cubic: ${sp.latex(cubic)}$')
plt.xlabel('u = x/d')
plt.legend(frameon=False)
plt.show()

In [ ]:
# How far off is the linear term? Percent error at a few values of u = x/d
for value in [0.01, 0.05, 0.1, 0.2]:
    exact = bracket.subs(u, value)
    print(f"u = {value}:  linear is off by {float(abs((linear.subs(u, value) - exact) / exact)) * 100:.2f}%")

## 2. A series inside a series: $1/(1 + \sin x)$ through $x^3$

The outer function is $\dfrac{1}{1+a}$; the inner one is $a = \sin x$.

In [ ]:
outer = (1/(1 + a)).series(a, 0, 4).removeO()     # 1 - a + a^2 - a^3
inner = sp.sin(x).series(x, 0, 4).removeO()        # x - x^3/6
outer, inner

Substitute the inner series for $a$ and multiply everything out. Which of these terms can we trust?

In [ ]:
substituted = sp.expand(outer.subs(a, inner))
substituted

Keep only the terms through $x^3$, then check against SymPy's own series of $1/(1 + \sin x)$:

In [ ]:
ours = sum(substituted.coeff(x, k) * x**k for k in range(4))
direct = (1/(1 + sp.sin(x))).series(x, 0, 4)
ours, direct

Your turn (Felder 2.92(c)): the Gaussian. Check your whiteboard answer for $\int_0^1 e^{-x^2}\,dx$ from the series through $x^8$
against the computer's value. Hint: `sp.exp(-x**2).series(x, 0, 10).removeO()`, then `sp.integrate(..., (x, 0, 1))`;
for the computer's value, `sp.N(sp.integrate(sp.exp(-x**2), (x, 0, 1)))`.

## 3. Taylor about a point

Pick a function, a center, and the number of terms. The dot is the point where we evaluate; the printout gives the error there.
Start with $\cos x$, center 0, and slide the number of terms up. Then switch the center to 3.

In [ ]:
functions = {'cos(x)': sp.cos(x), 'e^x': sp.exp(x), 'sin(x)': sp.sin(x)}
xs = np.linspace(-1, 6, 500)

def values(expression):
    return 0 * xs + sp.lambdify(x, expression)(xs)   # numbers to plot; the 0*xs turns a constant into a flat line

def plot_taylor(function='cos(x)', center=0, n_terms=1, x_eval=3.5):
    f = functions[function]
    # the Taylor formula: f^(k)(center)/k! (x - center)^k, keeping the non-zero terms
    terms = [f.diff(x, k).subs(x, center) / sp.factorial(k) * (x - center)**k for k in range(30)]
    terms = [term for term in terms if term != 0][:n_terms]
    approximation = sum(terms)
    exact = float(f.subs(x, x_eval))
    approx = float(approximation.subs(x, x_eval))
    plt.figure()
    plt.plot(xs, values(f), 'k--', label=function)
    plt.plot(xs, values(approximation), 'purple', linewidth=3, label=f'{n_terms} term(s) about x = {center}')
    plt.plot([x_eval], [exact], 'ko')
    plt.plot([x_eval], [approx], 'o', color='purple')
    plt.axvline(center, color='grey', linewidth=0.5)
    plt.ylim(-3, 3) if function != 'e^x' else plt.ylim(-5, 150)
    plt.legend(loc='upper left', frameon=False)
    plt.show()
    print(f"at x = {x_eval}: {function} = {exact:.4f}, the approximation gives {approx:.4f}, "
          f"off by {abs(approx - exact) / abs(exact) * 100:.2f}%")

interact(plot_taylor, function=list(functions), center=[0, 3], n_terms=(1, 12), x_eval=(2.0, 5.0, 0.1))

## What you found

1. The crystal: which term comes after $-4u$, and how big is the linear term's error at $u = 0.05$?

*(your answer here)*

2. $1/(1 + \sin x)$: which terms of the substituted product did you throw away, and why?

*(your answer here)*

3. $\cos(3.5)$ to 1%: how many terms about 0, and about 3? What does that say about where to center a series?

*(your answer here)*